In [1]:
import pyspark
print(pyspark.__version__)

3.5.1


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("data_skew_homework").getOrCreate()
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/30 20:56:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql import functions as F

In [4]:
users_df = spark.range(100_000).withColumnRenamed("id", "user_id")
users_df = users_df.withColumn(
    "name",
    F.concat(F.lit("user_"), F.col("user_id").cast("string"))
)
users_df = users_df.withColumn(
    "email",
    F.concat(F.lit("user_"), F.col("user_id").cast("string"), F.lit("@test.com"))
)
users_df = users_df.withColumn(
    "phone_number",
    F.concat(F.lit("+7"), (F.rand()*(9_999_999_999-1_000_000_000)+1_000_000_000).cast("bigint"))
)
users_df = users_df.withColumn(
    "signup_date", F.date_sub(F.current_date(), (F.rand()*100).cast("int"))
)

In [5]:
users_df.show()

+-------+-------+----------------+------------+-----------+
|user_id|   name|           email|phone_number|signup_date|
+-------+-------+----------------+------------+-----------+
|      0| user_0| user_0@test.com|+78688066229| 2026-07-07|
|      1| user_1| user_1@test.com|+75361509571| 2026-06-29|
|      2| user_2| user_2@test.com|+77547131553| 2026-08-08|
|      3| user_3| user_3@test.com|+76609536551| 2026-08-04|
|      4| user_4| user_4@test.com|+72163269786| 2026-06-26|
|      5| user_5| user_5@test.com|+71472443393| 2026-06-15|
|      6| user_6| user_6@test.com|+78505116091| 2026-08-03|
|      7| user_7| user_7@test.com|+79910800207| 2026-07-16|
|      8| user_8| user_8@test.com|+78591922676| 2026-08-26|
|      9| user_9| user_9@test.com|+79564682645| 2026-07-23|
|     10|user_10|user_10@test.com|+73995178946| 2026-07-10|
|     11|user_11|user_11@test.com|+76258749495| 2026-07-14|
|     12|user_12|user_12@test.com|+79895057721| 2026-08-16|
|     13|user_13|user_13@test.com|+78975

In [6]:
users_df = users_df.coalesce(1)

In [7]:
users_df.write.csv("users.csv", header=True, mode="overwrite")

In [8]:
event_types = F.array(F.lit("click"), F.lit("view"), F.lit("purchase"), F.lit("login"), F.lit("logout"))
device_types = F.array(F.lit("mobile"), F.lit("desktop"), F.lit("tablet"))

hot_events_df = spark.range(49_500_000).withColumnRenamed("id", "event_id")
hot_events_df = hot_events_df.withColumns({
    "user_id": (F.rand() * 3).cast("int"),
    "event_timestamp": (F.date_sub(F.current_date(), (F.rand()*100).cast("int"))),
    "ip_address": (F.concat(
        (F.rand()*255).cast("int"), F.lit("."), 
        (F.rand()*255).cast("int"), F.lit("."),
        (F.rand()*255).cast("int"), F.lit("."),
        (F.rand()*255).cast("int"),
    )),
    "event_type": (F.element_at(event_types, (F.rand() * 5 + 1).cast("int"))),
    "device_type": (F.element_at(device_types, (F.rand() * 3 + 1).cast("int"))),
    "session_id": ((F.rand()*(9_999_999_999-1_000_000_000)+1_000_000_000).cast("bigint")),
})

In [9]:
hot_events_df.show()

+--------+-------+---------------+---------------+----------+-----------+----------+
|event_id|user_id|event_timestamp|     ip_address|event_type|device_type|session_id|
+--------+-------+---------------+---------------+----------+-----------+----------+
|       0|      2|     2026-06-16| 179.146.32.213|     login|     tablet|3418690995|
|       1|      0|     2026-07-11|152.201.193.152|    logout|     mobile|7513636812|
|       2|      0|     2026-08-12| 24.216.213.114|  purchase|     tablet|9398503607|
|       3|      1|     2026-07-10|  166.70.27.192|     click|     tablet|5147524925|
|       4|      2|     2026-06-06| 166.218.32.249|     login|     tablet|3018743433|
|       5|      2|     2026-06-07|    16.17.48.87|     login|    desktop|8738502842|
|       6|      1|     2026-06-28|    16.30.5.159|  purchase|    desktop|6588379127|
|       7|      1|     2026-07-08|   51.230.2.187|    logout|     tablet|7038274485|
|       8|      1|     2026-08-28|  62.207.35.178|     click|    

In [10]:
cold_events_df = spark.range(49_500_000, 50_000_000).withColumnRenamed("id", "event_id")
cold_events_df = cold_events_df.withColumns({
    "user_id": (F.rand() * (100_000-3)+3).cast("int"),
    "event_timestamp": (F.date_sub(F.current_date(), (F.rand()*1000).cast("int"))),
    "ip_address": (F.concat(
        (F.rand()*255).cast("int"), F.lit("."), 
        (F.rand()*255).cast("int"), F.lit("."),
        (F.rand()*255).cast("int"), F.lit("."),
        (F.rand()*255
        ).cast("int"),
    )),
    "event_type": (F.element_at(event_types, (F.rand() * 5 + 1).cast("int"))),
    "device_type": (F.element_at(device_types, (F.rand() * 3 + 1).cast("int"))),
    "session_id": ((F.rand()*(9_999_999_999-1_000_000_000)+1_000_000_000).cast("bigint")),
})

In [11]:
cold_events_df.show()

+--------+-------+---------------+---------------+----------+-----------+----------+
|event_id|user_id|event_timestamp|     ip_address|event_type|device_type|session_id|
+--------+-------+---------------+---------------+----------+-----------+----------+
|49500000|  73433|     2024-01-10|  19.179.25.167|      view|    desktop|4340442961|
|49500001|  95839|     2024-01-09|  128.96.21.190|     click|     tablet|5828053577|
|49500002|  29352|     2025-01-18|    29.32.11.23|  purchase|     tablet|7461083816|
|49500003|  18699|     2024-09-19|182.115.189.213|     click|     tablet|7191696456|
|49500004|  57117|     2025-09-14|   97.26.195.81|    logout|     mobile|7283980348|
|49500005|  63900|     2024-09-06|121.130.223.246|  purchase|     mobile|4636424656|
|49500006|  98462|     2024-02-08| 216.251.56.101|  purchase|     mobile|5119103618|
|49500007|  98595|     2025-10-22|  80.96.213.111|  purchase|     tablet|6612847609|
|49500008|  32512|     2025-08-15|  56.167.141.58|     click|    

In [12]:
events_df = hot_events_df.unionByName(cold_events_df)

In [13]:
events_df = events_df.coalesce(1)

In [14]:
events_df.write.json("events.json", mode="overwrite")

In [15]:
spark.stop()

### Another dataset gen way

In [ ]:
# df_skewed = spark.range(0, 50_000_000, numPartitions=100) \
#     .withColumn("join_key", F.when(F.rand() < 0.98, F.lit(1))
#                             .otherwise(F.floor(F.rand() * 99 + 2).cast("int"))) \
#     .withColumn("skewed_val", F.concat(F.lit("large_data_"), F.col("id")))
# df_lookup = spark.range(1, 101) \
#     .withColumnRenamed("id", "join_key") \
#     .withColumn("key_description", F.concat(F.lit("Desc_"), F.col("join_key")))
# joined_df = df_skewed.join(df_lookup, on="join_key", how="inner")
# joined_df.write.format("noop").mode("overwrite").save()